In [1]:
from llama_cpp import Llama


llm = Llama(
    model_path="/home/migueldcarvalho/Projetos/Chatbot/modelos_locais/Qwen3-4B-Instruct-2507-UD-Q4_K_XL.gguf", 
    n_gpu_layers=-1, 
    verbose=True
)
print("\nModelo carregado com sucesso na GPU!")

llama_model_loader: loaded meta data with 42 key-value pairs and 398 tensors from /home/migueldcarvalho/Projetos/Chatbot/modelos_locais/Qwen3-4B-Instruct-2507-UD-Q4_K_XL.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen3-4B-Instruct-2507
llama_model_loader: - kv   3:                            general.version str              = 2507
llama_model_loader: - kv   4:                           general.finetune str              = Instruct
llama_model_loader: - kv   5:                           general.basename str              = Qwen3-4B-Instruct-2507
llama_model_loader: - kv   6:                       general.quantized_by str


Modelo carregado com sucesso na GPU!


Using gguf chat template: {%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0].role == 'system' %}
        {{- messages[0].content + '\n\n' }}
    {%- endif %}
    {{- "# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {{- '<|im_start|>system\n' + messages[0].content + '<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- set ns = namespace(multi_step_tool=true, last_query_index=messages|length - 1) %}
{%- for message in messages[::-1] %}
    {%- set i

In [5]:
system = '''    
### IDENTITY
    You are an expert Data Engineer and Data Analyst, specializing in writing advanced SQL queries to answer analytical questions.

    ### OBJECTIVE
    Generate valid, highly efficient SQL code to answer the user's question based strictly on the provided CSV metadata.

    ### STRICT RULES
      1. ONLY output valid SQL code enclosed within <code> </code> tags.
      2. Absolutely NO markdown formatting (like ```python), NO greetings, NO explanations, and NO conversational text.
      3. TABLE NAME: You MUST query from the table named "dt_table". Example: FROM dt_table
      4. STRICT SCHEMA ADHERENCE: Only use column names exactly as they appear in the Schema. Do not hallucinate columns.

    ### ACTUAL INPUT

      <METADATA>
      Description: {dataset_description}
      Dimensions: {rows} rows x {columns} columns
      Schema (Columns and Data Types): {column_type_dictionary}
      Statistical Summary: {basic_statistical_analysis}
      </METADATA>

      <USER_QUESTION>
      {user_question}
      </USER_QUESTION>'''

user = ''' 
    <METADATA>
    Description: {dataset_description}
    Dimensions: {rows} rows x {columns} columns
    Schema (Columns and Data Types): {column_type_dictionary}
    Statistical Summary: {basic_statistical_analysis}
    </METADATA>
    
    <USER_QUESTION>
    {user_question}
    </USER_QUESTION>
    '''

llm.create_chat_completion(
      messages = [
          {"role": "system", 'content':system},
          {
              "role": "user",
              "content": user
          }
      ]
)


Llama.generate: 237 prefix-match hit, remaining 75 prompt tokens to eval
llama_perf_context_print:        load time =     658.81 ms
llama_perf_context_print: prompt eval time =    1596.79 ms /    75 tokens (   21.29 ms per token,    46.97 tokens per second)
llama_perf_context_print:        eval time =    4378.24 ms /    50 runs   (   87.56 ms per token,    11.42 tokens per second)
llama_perf_context_print:       total time =    6007.86 ms /   125 tokens
llama_perf_context_print:    graphs reused =         47


{'id': 'chatcmpl-c6a6db42-4774-4d8c-8e67-74e17319bf66',
 'object': 'chat.completion',
 'created': 1772761090,
 'model': '/home/migueldcarvalho/Projetos/Chatbot/modelos_locais/Qwen3-4B-Instruct-2507-UD-Q4_K_XL.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': '<code>\nSELECT \n    "column1",\n    "column2",\n    "column3"\nFROM \n    dt_table\nWHERE \n    "column1" IS NOT NULL\nORDER BY \n    "column1" DESC;\n</code>'},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 312, 'completion_tokens': 50, 'total_tokens': 362}}

In [2]:
from llama_cpp import Llama
import json

# ==========================================
# 1. INICIALIZAÇÃO DO MODELO
# ==========================================
# n_gpu_layers=-1 joga todas as camadas do modelo para a sua RTX 4050
# n_ctx=2048 define o tamanho máximo da janela de contexto (ajuste conforme os metadados)
llm = Llama(
    model_path="./Qwen3-4B-Instruct-2507-Q4_K_M.gguf", # Caminho para o seu modelo GGUF
    n_gpu_layers=-1, 
    n_ctx=2048,      
    verbose=False    # Deixe True se quiser ver o uso de memória no terminal
)

# Simulando suas tabelas
descricoes_tabelas = """
1. Tabela 'vendas': Contém o histórico de transações, valor_total, id_cliente, data_venda.
2. Tabela 'produtos': Contém id_produto, nome_produto, categoria, preco_unitario.
3. Tabela 'clientes': Contém id_cliente, nome_cliente, email, estado.
"""

pergunta_usuario = "Quais foram os produtos mais caros vendidos ontem?"

# ==========================================
# PASSO 1: ROTEAMENTO (Seleção da Tabela)
# ==========================================
print("--- Iniciando Passo 1: Roteamento ---")

prompt_roteamento = f"""<|im_start|>system
Você é um roteador de banco de dados. Sua única tarefa é ler a pergunta do usuário e as descrições das tabelas, e retornar APENAS o nome da tabela que contém as informações necessárias para criar a query SQL.
<|im_end|>
<|im_start|>user
Tabelas disponíveis:
{descricoes_tabelas}

Pergunta: {pergunta_usuario}
<|im_end|>
<|im_start|>assistant
"""

# Forçando a saída para ser um JSON estrito contendo apenas o nome da tabela
schema_roteamento = {
    "type": "object",
    "properties": {
        "tabela_escolhida": {
            "type": "string",
            "enum": ["vendas", "produtos", "clientes"] # O modelo SÓ pode escolher uma dessas
        }
    },
    "required": ["tabela_escolhida"]
}

resposta_passo_1 = llm.create_completion(
    prompt_roteamento,
    max_tokens=15,
    response_format={"type": "json_object", "schema": schema_roteamento},
    temperature=0.1 # Temperatura baixa para máxima precisão
)

# Extraindo a tabela escolhida
resultado_json = json.loads(resposta_passo_1["choices"][0]["text"])
tabela_alvo = resultado_json["tabela_escolhida"]
print(f"Tabela selecionada pelo modelo: {tabela_alvo}\n")


# ==========================================
# PASSO 2: GERAÇÃO DO SQL
# ==========================================
print("--- Iniciando Passo 2: Geração de SQL ---")

# Em um cenário real, você buscaria os metadados completos (DDL) da tabela_alvo aqui.
# Simulando o DDL completo da tabela escolhida:
metadados_completos = """
CREATE TABLE produtos (
    id_produto INT PRIMARY KEY,
    nome_produto VARCHAR(255),
    categoria VARCHAR(100),
    preco_unitario DECIMAL(10, 2)
);
"""

prompt_sql = f"""<|im_start|>system
Você é um especialista em SQL. Retorne APENAS a query SQL válida baseada nos metadados fornecidos, sem explicações adicionais.
<|im_end|>
<|im_start|>user
Metadados da tabela '{tabela_alvo}':
{metadados_completos}

Pergunta: {pergunta_usuario}
<|im_end|>
<|im_start|>assistant
```sql
"""

resposta_passo_2 = llm.create_completion(
    prompt_sql,
    max_tokens=150,
    stop=["```", "<|im_end|>"], # Para a geração assim que o SQL terminar
    temperature=0.1
)

sql_gerado = resposta_passo_2["choices"][0]["text"].strip()
print("SQL Gerado:")
print(sql_gerado)

RuntimeError: Failed to load shared library '/home/migueldcarvalho/miniforge3/envs/chatbot/lib/python3.11/site-packages/llama_cpp/lib/libllama.so': libcudart.so.12: cannot open shared object file: No such file or directory